# Scouting Serie A : système de similarité de joueurs

Projet personnel : collecte de données publiques sur Sofascore (saison 2025/26 de Serie A), construction d'un système qui calcule, pour chaque joueur, ses profils statistiques les plus proches, puis publication d'un site où l'on choisit un joueur, on regarde son radar, on le compare à un deuxième joueur et on découvre ses profils similaires.

Cas d'usage visé : un club ne peut pas se payer un joueur, et cherche des joueurs qui lui ressemblent statistiquement, au même poste.

Démarche du notebook.

1. Récupération des identifiants Sofascore de la compétition et de la saison.
2. Collecte des statistiques de tous les joueurs de la saison (effectifs actuels, puis complétion avec les feuilles de match pour ne pas perdre les joueurs partis en cours de saison).
3. Nettoyage et filtrage sur un minimum de 900 minutes jouées.
4. Conversion des statistiques de volume en « par 90 minutes », pour comparer des rythmes et non des totaux.
5. Regroupement par poste large (Attaquant, Milieu, Défenseur, Gardien).
6. Normalisation des échelles, puis similarité cosinus entre joueurs d'un même poste, convertie en pourcentage de ressemblance.
7. Export des données et construction du site (GitHub Pages).

Une annexe facultative montre l'export vers une base MySQL.

**Comment lire ce notebook.** La collecte (parties 2 à 7) dure environ 35 minutes. Elle n'est lancée qu'une seule fois : si le fichier `serie_a_2526_sofascore_brut.csv` existe déjà, le notebook le relit et passe directement au calcul. Chaque ligne de code est commentée, pour qu'une autre personne puisse reprendre le travail.

## 1. Installation, imports et réglages

Les réglages sont regroupés ici. Le seuil de 900 minutes (environ 10 matchs complets) évite de comparer des joueurs sur un échantillon trop faible. L'interrupteur `A_COLLECTER` se règle tout seul : il vaut `True` uniquement si ni le fichier brut ni le fichier filtré n'existent. Pour forcer une nouvelle collecte, il suffit de supprimer ces deux fichiers.

In [1]:
!pip install cloudscraper scikit-learn --quiet

In [2]:
import os  # pour tester l'existence des fichiers et créer des dossiers
import json  # pour écrire les données du site au format JSON
import time  # pour espacer les requêtes et ne pas surcharger l'API
import pandas as pd  # pour manipuler les données sous forme de tableaux (dataframes)
import numpy as np  # pour les calculs sur les colonnes

SEUIL_MINUTES = 900  # minutes minimales pour entrer dans l'analyse (environ 10 matchs complets)
FICHIER_BRUT = "serie_a_2526_sofascore_brut.csv"  # fichier de la collecte complète, avant filtrage
FICHIER_FILTRE = "serie_a_2526_sofascore.csv"  # fichier des joueurs retenus, base de travail
A_COLLECTER = not os.path.exists(FICHIER_BRUT) and not os.path.exists(FICHIER_FILTRE)  # True seulement si aucun des deux fichiers n'existe : la collecte n'a alors jamais été faite
print("Collecte à lancer :", A_COLLECTER)  # affiche le choix fait : False veut dire que le fichier brut existe déjà

Collecte à lancer : False


In [3]:
if A_COLLECTER:  # la collecte n'est lancée qu'une seule fois
    import pandas as pd  # pour manipuler les données sous forme de tableaux (dataframes)
    import re  # pour nettoyer du texte si besoin
    import time  # pour espacer les requêtes et ne pas surcharger l'API
    import cloudscraper  # librairie capable de contourner la protection anti-bot Cloudflare utilisée par Sofascore

    scraper = cloudscraper.create_scraper()  # crée un "scraper" qui se comporte comme un vrai navigateur face à la protection Cloudflare, contrairement à requests seul

## 2. Identification de la compétition et de la saison

Sofascore n'a pas de documentation publique officielle. Les identifiants (compétition, saison, équipe, joueur) ont été retrouvés par exploration de l'API interne du site.

In [4]:
if A_COLLECTER:  # collecte uniquement
    serie_a = {"id": 23}  # identifiant de la Serie A (Italie) sur Sofascore, connu à l'avance : plus fiable que la recherche, que Sofascore refuse parfois

    print(f"Serie A (Italie), id = {serie_a['id']}")  # affiche l'identifiant utilisé, on attend 23

In [5]:
if A_COLLECTER:  # collecte uniquement
    url_seasons = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/seasons"  # endpoint qui liste toutes les saisons disponibles pour cette compétition

    response_seasons = scraper.get(url_seasons)  # requête via le scraper

    season = {"id": 76457}  # identifiant de la saison 25/26, connu à l'avance, utilisé si Sofascore ne renvoie pas la liste

    if response_seasons.status_code == 200:  # si Sofascore répond normalement, on vérifie l'identifiant dans la liste
        for item in response_seasons.json().get("seasons", []):  # on parcourt chaque saison disponible
            if item["year"] == "25/26":  # on cherche la saison 2025/2026
                season = item  # on garde la saison trouvée
                break  # inutile de continuer une fois trouvée
    else:  # sinon Sofascore bloque la requête
        print("Réponse Sofascore :", response_seasons.status_code)  # affiche le code d'erreur pour comprendre le blocage (403 = accès refusé)

    print(f"Saison 25/26, id = {season['id']}")  # affiche l'identifiant de la saison, on attend 76457

## 3. Liste des 20 équipes de la saison

Récupérée via le classement final, qui garantit d'avoir les 20 équipes ayant réellement disputé la saison.

In [6]:
if A_COLLECTER:  # collecte uniquement
    url_standings = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/standings/total"  # endpoint du classement complet de la saison

    response_standings = scraper.get(url_standings)  # requête via le scraper

    data_standings = response_standings.json()  # transforme en dictionnaire Python

    teams = []  # liste qui contiendra un dictionnaire (nom + id) par équipe
    for row in data_standings["standings"][0]["rows"]:  # on parcourt chaque ligne du classement, une par équipe
        teams.append({"name": row["team"]["name"], "id": row["team"]["id"]})  # on garde uniquement le nom et l'id, les seules infos utiles pour la suite

    print(f"{len(teams)} équipes récupérées")  # contrôle : on attend 20 équipes

## 4. Colonnes de statistiques à conserver

Sélection faite en amont pour éviter de récupérer une centaine de colonnes brutes peu exploitables. Les colonnes sont regroupées par catégorie pour la lisibilité.

In [7]:
colonnes_identite = [  # colonnes qui identifient le joueur
    "id",  # id Sofascore du joueur
    "name",  # nom complet du joueur
    "position",  # poste en une lettre (G, D, M, F), utilisé ensuite pour le regroupement large
    "dateOfBirth",  # utile pour distinguer les homonymes
]

colonnes_temps_de_jeu = [  # colonnes de temps de jeu
    "minutesPlayed",  # sert à appliquer le seuil minimum de 900 minutes
    "appearances",  # nombre de matchs joués
    "matchesStarted",  # nombre de titularisations
]

colonnes_offensives = [  # colonnes offensives
    "goals",  # buts marqués
    "expectedGoals",  # xG, propre au modèle Sofascore
    "assists",  # passes décisives
    "expectedAssists",  # xA
    "bigChancesCreated",  # grosses occasions créées pour un coéquipier
    "totalShots",  # volume de tirs
    "shotsOnTarget",  # tirs cadrés
    "successfulDribbles",  # dribbles réussis
    "keyPasses",  # passes menant à un tir
]

colonnes_passes_possession = [  # colonnes de passes et de possession
    "accuratePasses",  # passes réussies, volume brut
    "totalPasses",  # passes tentées
    "accuratePassesPercentage",  # taux de réussite des passes
    "accurateLongBalls",  # longs ballons réussis
    "touches",  # nombre de touches de balle
]

colonnes_defensives = [  # colonnes défensives
    "tackles",  # tacles tentés
    "tacklesWon",  # tacles réussis
    "interceptions",  # interceptions
    "clearances",  # dégagements
    "blockedShots",  # tirs contrés
    "aerialDuelsWon",  # duels aériens gagnés
    "aerialDuelsWonPercentage",  # taux de réussite dans les airs
    "groundDuelsWon",  # duels au sol gagnés
]

colonnes_discipline = [  # colonnes de discipline
    "yellowCards",  # cartons jaunes
    "redCards",  # cartons rouges
    "fouls",  # fautes commises
]

colonnes_gardien = [  # colonnes propres aux gardiens
    "saves",  # arrêts, pertinent uniquement pour les gardiens
    "cleanSheet",  # matchs sans encaisser de but
    "goalsConceded",  # buts encaissés
    "penaltySave",  # penalties arrêtés
]

colonnes_a_garder = (  # début de la fusion des catégories
    colonnes_identite  # identité
    + colonnes_temps_de_jeu  # plus le temps de jeu
    + colonnes_offensives  # plus l'offensif
    + colonnes_passes_possession  # plus les passes et la possession
    + colonnes_defensives  # plus le défensif
    + colonnes_discipline  # plus la discipline
    + colonnes_gardien  # plus les gardiens
)  # fusion de toutes les catégories en une seule liste, pour filtrer en une seule passe

In [8]:
def extraire_stats_joueur(player_info, stats_json):  # petite recette : transforme les données d'un joueur en une ligne de tableau
    # player_info : dictionnaire d'identité du joueur (id, nom, poste, date de naissance, équipe)
    # stats_json : dictionnaire complet renvoyé par l'endpoint statistics/overall pour ce joueur
    stats = stats_json.get("statistics", {})  # sous-dictionnaire des stats, vide par défaut pour éviter un crash si la clé manque

    ligne = {}  # dictionnaire qui contiendra une seule ligne de la future table (un joueur)
    ligne["id"] = player_info["id"]  # id Sofascore du joueur
    ligne["name"] = player_info["name"]  # nom complet
    ligne["position"] = player_info["position"]  # poste en une lettre
    ligne["dateOfBirth"] = player_info.get("dateOfBirth")  # date de naissance
    ligne["team"] = player_info["team"]["name"]  # nom de l'équipe

    for colonne in colonnes_a_garder:  # on parcourt toutes les colonnes de stats choisies
        if colonne not in ligne:  # on évite d'écraser les colonnes d'identité déjà ajoutées
            ligne[colonne] = stats.get(colonne)  # .get() renvoie None si la clé n'existe pas (ex: stats gardien chez un attaquant), plutôt que de planter

    return ligne  # dictionnaire complet pour ce joueur

## 5. Collecte des statistiques, effectifs actuels

Première passe : pour chaque équipe, on récupère son effectif actuel puis les statistiques individuelles de chaque joueur sur la saison 2025/26.

Limite de cette première passe : l'effectif « actuel » ne contient pas les joueurs partis après la saison (transferts, fins de contrat). Cette limite est corrigée à l'étape suivante.

Une pause de 2 secondes sépare chaque requête pour ne pas surcharger l'API. Cette cellule prend environ 15 à 20 minutes. **Contrôle attendu** : 387 joueurs récupérés.

In [9]:
if A_COLLECTER:  # collecte uniquement
    toutes_les_lignes = []  # liste qui contiendra un dictionnaire par joueur, pour toute la Serie A

    for equipe in teams:  # on parcourt les 20 équipes du classement
        url_roster = f"https://api.sofascore.com/api/v1/team/{equipe['id']}/players"  # effectif actuel de cette équipe

        response_roster = scraper.get(url_roster)  # requête via le scraper
        time.sleep(2)  # pause pour ne pas surcharger l'API

        if response_roster.status_code != 200:  # sécurité : si le roster échoue, on log et on passe à l'équipe suivante
            print(f"Erreur roster pour {equipe['name']} : {response_roster.status_code}")  # affiche l'équipe en cause et le code d'erreur
            continue  # passe à l'élément suivant

        data_roster = response_roster.json()  # transforme en dictionnaire Python

        for joueur_roster in data_roster["players"]:  # on parcourt chaque joueur de l'effectif
            player_info = joueur_roster["player"]  # sous-dictionnaire des infos du joueur

            url_stats = f"https://api.sofascore.com/api/v1/player/{player_info['id']}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # stats individuelles du joueur pour la saison 25/26

            response_stats = scraper.get(url_stats)  # requête via le scraper
            time.sleep(2)  # pause

            if response_stats.status_code != 200:  # cas attendu : joueur arrivé cet été, aucune stat pour la saison 25/26 avec ce club
                continue  # passe à l'élément suivant

            stats_json = response_stats.json()  # transforme en dictionnaire Python

            ligne = extraire_stats_joueur(player_info, stats_json)  # on filtre uniquement les colonnes choisies
            toutes_les_lignes.append(ligne)  # on ajoute ce joueur à la liste finale

        print(f"{equipe['name']} terminé, {len(data_roster['players'])} joueurs traités")  # affiche l'avancement équipe par équipe

    print(f"Total première passe : {len(toutes_les_lignes)} joueurs récupérés")  # contrôle : on attend 387 joueurs

## 6. Complétion : joueurs partis en cours de saison

Pour retrouver les joueurs présents pendant la saison mais absents des effectifs actuels, on reconstitue la liste de tous les joueurs qui ont figuré dans une feuille de match (titulaires et remplaçants confondus) sur les 38 journées.

Cette partie fait environ 420 requêtes (38 journées et environ 380 matchs) et prend environ 14 minutes. **Contrôles attendus** : 772 joueurs uniques dans les feuilles de match, 385 joueurs à compléter, puis 586 joueurs au total.

In [10]:
if A_COLLECTER:  # collecte uniquement
    ids_joueurs_saison = set()  # ensemble (sans doublons) qui contiendra tous les id de joueurs ayant figuré dans une feuille de match

    for journee in range(1, 39):  # 38 journées de Serie A
        url_events = f"https://api.sofascore.com/api/v1/unique-tournament/{serie_a['id']}/season/{season['id']}/events/round/{journee}"  # matchs de cette journée

        response_events = scraper.get(url_events)  # requête via le scraper
        time.sleep(2)  # pause

        if response_events.status_code != 200:  # sécurité, on log et on continue
            print(f"Erreur journée {journee} : {response_events.status_code}")  # affiche la journée en cause et le code d'erreur
            continue  # passe à l'élément suivant

        data_events = response_events.json()  # transforme en dictionnaire Python

        for match in data_events["events"]:  # on parcourt chaque match de cette journée
            url_lineups = f"https://api.sofascore.com/api/v1/event/{match['id']}/lineups"  # compositions de ce match

            response_lineups = scraper.get(url_lineups)  # requête via le scraper
            time.sleep(2)  # pause

            if response_lineups.status_code != 200:  # certains matchs peuvent ne pas avoir de compositions enregistrées
                continue  # passe à l'élément suivant

            data_lineups = response_lineups.json()  # transforme en dictionnaire Python

            for cote in ["home", "away"]:  # on traite les deux équipes du match
                if cote not in data_lineups or "players" not in data_lineups[cote]:  # sécurité pour les matchs incomplets
                    continue  # passe à l'élément suivant

                for joueur in data_lineups[cote]["players"]:  # titulaires + remplaçants confondus
                    ids_joueurs_saison.add(joueur["player"]["id"])  # ajout à l'ensemble, les doublons sont ignorés automatiquement

        print(f"Journée {journee} terminée, {len(ids_joueurs_saison)} id uniques cumulés")  # affiche l'avancement journée par journée

    print(f"Total : {len(ids_joueurs_saison)} joueurs uniques ayant figuré dans une feuille de match en 25/26")  # contrôle : on attend 772 joueurs

In [11]:
if A_COLLECTER:  # collecte uniquement
    ids_deja_recuperes = set(pd.DataFrame(toutes_les_lignes)["id"])  # id déjà présents dans notre collecte issue des effectifs actuels

    ids_manquants = ids_joueurs_saison - ids_deja_recuperes  # différence entre les deux ensembles, les joueurs à compléter

    print(f"{len(ids_manquants)} joueurs manquants à compléter")  # contrôle : on attend 385 joueurs à compléter

In [12]:
if A_COLLECTER:  # collecte uniquement
    for id_joueur in ids_manquants:  # on parcourt les id manquants
        url_stats = f"https://api.sofascore.com/api/v1/player/{id_joueur}/unique-tournament/{serie_a['id']}/season/{season['id']}/statistics/overall"  # stats individuelles pour la saison 25/26

        response_stats = scraper.get(url_stats)  # requête via le scraper
        time.sleep(2)  # pause

        if response_stats.status_code != 200:  # cas attendu : joueur jamais entré en jeu, donc sous le seuil de 900 minutes de toute façon
            continue  # passe à l'élément suivant

        data_stats = response_stats.json()  # transforme en dictionnaire Python

        url_player_info = f"https://api.sofascore.com/api/v1/player/{id_joueur}"  # infos d'identité du joueur
        response_player_info = scraper.get(url_player_info)  # requête via le scraper
        time.sleep(2)  # pause

        if response_player_info.status_code != 200:  # sécurité
            continue  # passe à l'élément suivant

        data_player_info = response_player_info.json()["player"]  # extrait directement le sous-dictionnaire "player"

        player_info_complet = {  # début du dictionnaire d'identité reconstruit
            "id": data_player_info["id"],  # id Sofascore
            "name": data_player_info["name"],  # nom complet
            "position": data_player_info["position"],  # poste en une lettre
            "dateOfBirth": data_player_info.get("dateOfBirth"),  # date de naissance
            "team": {"name": data_stats.get("team", {}).get("name")},  # équipe prise depuis les stats de la saison 25/26 (pas depuis le profil joueur, qui donnerait l'équipe actuelle, potentiellement différente)
        }  # dictionnaire reconstruit au même format que celui issu du roster, pour réutiliser extraire_stats_joueur sans modification

        ligne = extraire_stats_joueur(player_info_complet, data_stats)  # même fonction d'extraction que pour la première passe
        toutes_les_lignes.append(ligne)  # ajout direct à la liste existante

    print(f"Total après complétion : {len(toutes_les_lignes)} joueurs")  # contrôle : on attend 586 joueurs

## 7. Nettoyage, dédoublonnage et sauvegarde

Le résultat brut est sauvegardé avant tout filtrage : on peut ainsi refaire les étapes suivantes sans refaire la collecte. Aux lancements suivants, le fichier est simplement relu.

**Contrôle attendu** : 586 joueurs et 37 colonnes.

In [13]:
if A_COLLECTER:  # si on vient de collecter, on assemble les résultats
    df = pd.DataFrame(toutes_les_lignes)  # reconstruit le tableau à partir de la liste complète (première passe et complétion)
    df = df.drop_duplicates(subset="id", keep="last")  # supprime les doublons éventuels sur l'identifiant du joueur
    df.to_csv(FICHIER_BRUT, index=False)  # sauvegarde le résultat brut avant filtrage, pour ne jamais avoir à tout refaire
elif os.path.exists(FICHIER_BRUT):  # sinon, si la collecte a déjà été faite
    df = pd.read_csv(FICHIER_BRUT)  # relit le fichier brut sauvegardé
else:  # sinon, il ne reste que le fichier filtré (339 joueurs déjà retenus)
    df = pd.read_csv(FICHIER_FILTRE)  # relit le fichier filtré, le filtrage à 900 minutes de l'étape suivante ne change alors rien

print(df.shape)  # contrôle : on attend (586, 37) avec le fichier brut, (339, 37) avec le fichier filtré seul

(339, 37)


## 8. Filtrage sur le temps de jeu

Seuil retenu : 900 minutes jouées au minimum (environ 10 matchs complets), pour ne garder que des statistiques fiables et éviter les échantillons trop faibles. Un joueur qui entre souvent en cours de match mais joue peu au total est donc absent de l'analyse, c'est une limite assumée.

**Contrôle attendu** : 339 joueurs retenus.

In [14]:
df_filtre = df[df["minutesPlayed"] >= SEUIL_MINUTES].copy()  # on ne garde que les joueurs ayant joué au moins le seuil de minutes

print(df_filtre.shape)  # contrôle : on attend (339, 37)

df_filtre.to_csv(FICHIER_FILTRE, index=False)  # sauvegarde le résultat filtré, notre base de travail pour la suite

(339, 37)


## 9. Regroupement par poste large

Choix d'un découpage large (Attaquant, Milieu, Défenseur, Gardien) plutôt que fin : un même joueur peut occuper plusieurs rôles précis au sein d'une même grande catégorie au cours d'une saison. Les gardiens sont gardés à part, leurs statistiques n'ayant pas de recouvrement avec le jeu de champ.

Conséquence à garder en tête : un ailier et un avant centre portent tous deux le code F de Sofascore. Ils sont donc comparés ensemble.

In [15]:
mapping_postes = {  # début de la correspondance des postes
    "G": "Gardien",  # G devient Gardien
    "D": "Défenseur",  # D devient Défenseur
    "M": "Milieu",  # M devient Milieu
    "F": "Attaquant",  # F devient Attaquant
}  # correspondance entre le code Sofascore et le regroupement large utilisé

df_filtre["poste_large"] = df_filtre["position"].map(mapping_postes)  # nouvelle colonne, sans toucher à la colonne "position" d'origine

print(df_filtre["poste_large"].value_counts())  # contrôle : on attend 135 milieux, 105 défenseurs, 75 attaquants et 24 gardiens

poste_large
Milieu       135
Défenseur    105
Attaquant     75
Gardien       24
Name: count, dtype: int64


## 10. Conversion en « par 90 minutes »

Un joueur ayant joué 900 minutes et un autre 3000 minutes ne sont pas comparables sur des totaux de saison (buts, tacles), même avec un profil de jeu identique. On convertit donc les statistiques de volume en taux « par 90 minutes », pour comparer des rythmes de production plutôt que des cumuls.

Les statistiques déjà exprimées en pourcentage, ainsi que les clean sheets (un événement par match entier, pas un taux continu), restent en l'état.

In [16]:
colonnes_pourcentage = [  # colonnes déjà exprimées en pourcentage
    "accuratePassesPercentage",  # déjà un taux
    "aerialDuelsWonPercentage",  # déjà un taux
]

colonnes_a_exclure_du_per90 = (  # début de la liste des colonnes à ne pas convertir
    ["id", "name", "position", "dateOfBirth", "team", "poste_large"]  # colonnes d'identité
    + ["minutesPlayed", "appearances", "matchesStarted", "cleanSheet"]  # temps de jeu (sert de dénominateur) et cleanSheet (événement par match, pas un taux continu)
    + colonnes_pourcentage  # taux déjà exclus au-dessus
)

colonnes_a_convertir = [col for col in df_filtre.columns if col not in colonnes_a_exclure_du_per90]  # toutes les colonnes restantes sont des volumes à convertir

for colonne in colonnes_a_convertir:  # on parcourt chaque colonne de volume
    df_filtre[colonne + "_per90"] = df_filtre[colonne] / df_filtre["minutesPlayed"] * 90  # règle de trois : ramène le total de saison à un rythme sur un match complet

print(df_filtre[["name", "goals", "minutesPlayed", "goals_per90"]].head())  # contrôle : Lautaro Martínez doit afficher 17 buts, 2178 minutes et 0,70 but par 90 minutes

               name  goals  minutesPlayed  goals_per90
0  Lautaro Martínez     17           2178     0.702479
1     Marcus Thuram     13           1913     0.611605
2   Ange-Yoan Bonny      5           1155     0.389610
3      Pio Esposito      7           1572     0.400763
4  Hakan Çalhanoğlu      9           1649     0.491207


## 11. Colonnes retenues pour le calcul de similarité, par poste

Un attaquant et un défenseur ne se comparent pas sur les mêmes statistiques. Chaque poste a son propre jeu de colonnes, choisi pour refléter les dimensions pertinentes de ce rôle (offensif, passes et possession, défensif, selon le cas).

In [17]:
colonnes_attaque = [  # colonnes de comparaison des attaquants
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",  # buts, xG, passes décisives et xA
    "totalShots_per90", "shotsOnTarget_per90", "bigChancesCreated_per90",  # tirs, tirs cadrés, grosses occasions créées
    "successfulDribbles_per90", "keyPasses_per90", "touches_per90",  # dribbles, passes clés, touches
]

colonnes_milieu = [  # colonnes de comparaison des milieux
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",  # buts, xG, passes décisives et xA
    "keyPasses_per90", "accuratePasses_per90", "accuratePassesPercentage",  # passes clés, passes réussies et taux de réussite
    "accurateLongBalls_per90", "successfulDribbles_per90", "touches_per90",  # longs ballons, dribbles, touches
    "tackles_per90", "interceptions_per90",  # tacles et interceptions
    "groundDuelsWon_per90", "aerialDuelsWon_per90",  # duels au sol et aériens gagnés
]

colonnes_defense = [  # colonnes de comparaison des défenseurs
    "tackles_per90", "tacklesWon_per90", "interceptions_per90", "clearances_per90",  # tacles, tacles réussis, interceptions, dégagements
    "blockedShots_per90", "aerialDuelsWon_per90", "aerialDuelsWonPercentage",  # tirs contrés, duels aériens gagnés et taux de réussite
    "groundDuelsWon_per90", "accuratePasses_per90", "accuratePassesPercentage",  # duels au sol, passes réussies et taux de réussite
    "goals_per90", "expectedGoals_per90", "assists_per90", "expectedAssists_per90",  # buts, xG, passes décisives et xA
    "successfulDribbles_per90", "keyPasses_per90", "touches_per90",  # dribbles, passes clés, touches
]

colonnes_gardien = [  # colonnes de comparaison des gardiens
    "saves_per90", "goalsConceded_per90", "penaltySave_per90",  # arrêts, buts encaissés, penalties arrêtés
    "cleanSheet",  # volume brut, pas per90 (voir remarque section 10)
]

## 12. Normalisation des échelles

Utilisation de `StandardScaler` (moyenne 0, écart type 1), appliqué séparément par poste : un défenseur doit être comparé à l'échelle des autres défenseurs, pas à celle de tous les joueurs mélangés. Sans cette étape, une statistique à grande échelle comme les touches (environ 40 par match) écraserait une statistique à petite échelle comme les buts (moins de 1 par match).

In [18]:
from sklearn.preprocessing import StandardScaler  # centre-réduit chaque colonne (moyenne 0, écart-type 1)

def normaliser_groupe(df, colonnes):  # petite recette : normalise les colonnes d'un seul poste
    # df : sous-ensemble du DataFrame pour un seul poste
    # colonnes : colonnes de stats à normaliser pour ce poste précis
    scaler = StandardScaler()  # un nouveau scaler à chaque appel, pour ne pas mélanger les échelles entre postes
    valeurs_normalisees = scaler.fit_transform(df[colonnes])  # calcule et applique la normalisation
    df_normalise = pd.DataFrame(valeurs_normalisees, columns=[c + "_norm" for c in colonnes], index=df.index)  # même index que df, pour recoller sans décalage
    return pd.concat([df, df_normalise], axis=1)  # colle les colonnes normalisées à côté des colonnes d'origine

df_attaque = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Attaquant"], colonnes_attaque)  # attaquants normalisés
df_milieu = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Milieu"], colonnes_milieu)  # milieux normalisés
df_defense = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Défenseur"], colonnes_defense)  # défenseurs normalisés
df_gardien = normaliser_groupe(df_filtre[df_filtre["poste_large"] == "Gardien"], colonnes_gardien)  # gardiens normalisés

print(f"{len(df_attaque)} attaquants, {len(df_milieu)} milieux, {len(df_defense)} défenseurs, {len(df_gardien)} gardiens")  # contrôle : on attend 75, 135, 105 et 24

75 attaquants, 135 milieux, 105 défenseurs, 24 gardiens


## 13. Calcul de la similarité

Similarité cosinus entre les profils statistiques normalisés : elle compare la direction du profil plutôt que sa magnitude brute, donc deux joueurs de style proche mais de niveau de performance légèrement différent restent identifiés comme proches.

Le score (entre moins 1 et 1) est ensuite converti en pourcentage de ressemblance (0 à 100). Lecture : 100 veut dire un profil identique, 50 aucun lien entre les profils, 0 des profils opposés. Sur nos 75 attaquants, la moitié des paires de joueurs est sous 50 %, et seulement 4 % des paires dépassent 90 % : un score élevé a donc un vrai sens.

**Contrôle attendu** : après Lautaro Martínez lui même, Marcus Thuram à 97,7, Nikola Krstović à 92,9, Rafael Leão à 91,6, Donyell Malen à 89,3 et Dušan Vlahović à 87,5.

In [19]:
from sklearn.metrics.pairwise import cosine_similarity  # calcule la similarité cosinus entre toutes les paires de lignes d'une matrice

def calculer_similarite(df, colonnes):  # petite recette : matrice de similarité d'un poste
    # df : sous-table d'un seul poste, déjà normalisée
    # colonnes : colonnes de base (sans le suffixe _norm)
    colonnes_norm = [c + "_norm" for c in colonnes]  # reconstruit les noms des colonnes normalisées
    matrice_similarite = cosine_similarity(df[colonnes_norm])  # score de similarité entre chaque paire de joueurs
    return pd.DataFrame(matrice_similarite, index=df["name"], columns=df["name"])  # matrice lisible, joueurs en lignes et en colonnes

def convertir_en_pourcentage(df_similarite):  # petite recette : convertit les scores en pourcentages
    # df_similarite : matrice de similarité cosinus (valeurs entre -1 et 1)
    return ((df_similarite + 1) / 2 * 100).round(1)  # ramène l'intervalle [-1, 1] vers [0, 100]

similarite_attaque = calculer_similarite(df_attaque, colonnes_attaque)  # scores des attaquants
similarite_milieu = calculer_similarite(df_milieu, colonnes_milieu)  # scores des milieux
similarite_defense = calculer_similarite(df_defense, colonnes_defense)  # scores des défenseurs
similarite_gardien = calculer_similarite(df_gardien, colonnes_gardien)  # scores des gardiens

pourcentage_attaque = convertir_en_pourcentage(similarite_attaque)  # pourcentages des attaquants
pourcentage_milieu = convertir_en_pourcentage(similarite_milieu)  # pourcentages des milieux
pourcentage_defense = convertir_en_pourcentage(similarite_defense)  # pourcentages des défenseurs
pourcentage_gardien = convertir_en_pourcentage(similarite_gardien)  # pourcentages des gardiens

print(pourcentage_attaque["Lautaro Martínez"].sort_values(ascending=False).head(6))  # contrôle : après Lautaro lui même, Thuram 97,7, Krstović 92,9, Leão 91,6, Malen 89,3 et Vlahović 87,5

name
Lautaro Martínez    100.0
Marcus Thuram        97.7
Nikola Krstović      92.9
Rafael Leão          91.6
Donyell Malen        89.3
Dušan Vlahović       87.5
Name: Lautaro Martínez, dtype: float64


## 14. Fonction de recherche de joueurs similaires

Une petite fonction qui renvoie, pour un joueur, ses profils les plus proches avec leurs statistiques et leur pourcentage de ressemblance. C'est la même logique que le site, qui la reprend sous forme de page.

In [20]:
def chercher_joueurs_similaires(nom_joueur, top_n=10):  # petite recette : les joueurs les plus proches d'un joueur donné
    # nom_joueur : nom exact du joueur recherché, tel qu'il apparaît dans la colonne "name"
    # top_n : nombre de joueurs similaires à retourner

    ligne_joueur = df_filtre[df_filtre["name"] == nom_joueur]  # cherche la ligne correspondant au joueur demandé

    if ligne_joueur.empty:  # sécurité : nom introuvable
        print(f"Joueur '{nom_joueur}' introuvable, vérifie l'orthographe exacte")  # message d'aide
        return None  # rien à renvoyer

    poste = ligne_joueur["poste_large"].iloc[0]  # poste large du joueur, pour choisir la bonne matrice et la bonne table

    matrices_par_poste = {  # début de la correspondance poste, matrice, table et colonnes
        "Attaquant": (pourcentage_attaque, df_attaque, colonnes_attaque),  # attaquants
        "Milieu": (pourcentage_milieu, df_milieu, colonnes_milieu),  # milieux
        "Défenseur": (pourcentage_defense, df_defense, colonnes_defense),  # défenseurs
        "Gardien": (pourcentage_gardien, df_gardien, colonnes_gardien),  # gardiens
    }  # dictionnaire de correspondance poste -> (matrice de pourcentage, table de stats, colonnes utilisées)

    matrice_pourcentage, df_poste, colonnes_stats = matrices_par_poste[poste]  # récupère les 3 éléments correspondant au poste

    joueurs_proches = matrice_pourcentage[nom_joueur].sort_values(ascending=False).iloc[1:top_n + 1]  # scores triés, en excluant le joueur lui-même (toujours 100%)

    resultat = df_poste[df_poste["name"].isin(joueurs_proches.index)][["name", "team", "position"] + colonnes_stats].copy()  # stats des joueurs similaires

    resultat["pourcentage_ressemblance"] = resultat["name"].map(joueurs_proches)  # ajoute le pourcentage en dernière colonne

    return resultat.sort_values("pourcentage_ressemblance", ascending=False).reset_index(drop=True)  # tri final par ressemblance décroissante

chercher_joueurs_similaires("Lautaro Martínez", top_n=5)  # exemple : les 5 joueurs les plus proches de Lautaro Martínez

,name,team,position,goals_per90,expectedGoals_per90,assists_per90,expectedAssists_per90,totalShots_per90,shotsOnTarget_per90,bigChancesCreated_per90,successfulDribbles_per90,keyPasses_per90,touches_per90,pourcentage_ressemblance
0,Marcus Thuram,Inter,F,0.611605,0.443832,0.282279,0.120354,3.481443,1.364349,0.329326,0.799791,1.364349,39.095661,97.7
1,Nikola Krstović,Atalanta,F,0.503919,0.706772,0.251960,0.106895,4.988802,1.713326,0.251960,0.856663,1.058231,43.992161,92.9
2,Rafael Leão,AC Milan,F,0.435718,0.478143,0.145239,0.163224,3.050027,1.161915,0.484131,1.258741,1.113502,44.104357,91.6
3,Donyell Malen,AS Roma,F,0.852503,0.863184,0.121786,0.145694,4.262517,1.887686,0.121786,0.974290,0.487145,29.959405,89.3
4,Dušan Vlahović,Juventus,F,0.645492,0.652850,0.092213,0.110040,4.518443,1.659836,0.092213,0.276639,1.106557,32.274590,87.5


## 15. Configuration du radar

Le radar affiche, pour chaque poste, une sélection de statistiques par 90 minutes (les gardiens gardent des statistiques brutes pour les clean sheets). Chaque axe est ramené sur 100 par rapport à la valeur maximale observée parmi les joueurs du même poste : 100 veut dire « le meilleur de son poste sur cette statistique », pas « 100 pour 100 ».

Pour les gardiens, les buts encaissés sont une statistique défavorable : plus la valeur est haute, moins le profil est bon.

In [21]:
POSTES_RADAR = {  # pour chaque poste : le code Sofascore et les axes du radar (nom affiché : colonne)
    "Attaquant": {  # attaquants
        "code": "F",  # code du poste chez Sofascore
        "radar": {  # axes du radar
            "Buts /90": "goals_per90",  # buts
            "xG /90": "expectedGoals_per90",  # qualité des tirs
            "Passes D /90": "assists_per90",  # passes décisives
            "Tirs /90": "totalShots_per90",  # volume de tirs
            "Tirs cadrés /90": "shotsOnTarget_per90",  # tirs cadrés
            "Dribbles réussis /90": "successfulDribbles_per90",  # dribbles
            "Passes clés /90": "keyPasses_per90",  # passes menant à un tir
            "Duels sol /90": "groundDuelsWon_per90",  # duels au sol gagnés
        },
    },
    "Milieu": {  # milieux
        "code": "M",  # code du poste chez Sofascore
        "radar": {  # axes du radar
            "Buts /90": "goals_per90",  # buts
            "xG /90": "expectedGoals_per90",  # qualité des tirs
            "Passes D /90": "assists_per90",  # passes décisives
            "xA /90": "expectedAssists_per90",  # qualité des passes décisives
            "Grosses occasions /90": "bigChancesCreated_per90",  # grosses occasions créées
            "Tirs cadrés /90": "shotsOnTarget_per90",  # tirs cadrés
            "Passes clés /90": "keyPasses_per90",  # passes menant à un tir
            "Passes réussies /90": "accuratePasses_per90",  # passes réussies
            "Passes tentées /90": "totalPasses_per90",  # passes tentées
            "Longs ballons réussis /90": "accurateLongBalls_per90",  # longs ballons réussis
            "Dribbles réussis /90": "successfulDribbles_per90",  # dribbles
            "Touches /90": "touches_per90",  # touches de balle
            "Tacles /90": "tackles_per90",  # tacles
            "Interceptions /90": "interceptions_per90",  # interceptions
            "Duels sol /90": "groundDuelsWon_per90",  # duels au sol gagnés
            "Duels aériens /90": "aerialDuelsWon_per90",  # duels aériens gagnés
        },
    },
    "Défenseur": {  # défenseurs
        "code": "D",  # code du poste chez Sofascore
        "radar": {  # axes du radar
            "Buts /90": "goals_per90",  # buts
            "xG /90": "expectedGoals_per90",  # qualité des tirs
            "Passes D /90": "assists_per90",  # passes décisives
            "xA /90": "expectedAssists_per90",  # qualité des passes décisives
            "Tacles /90": "tackles_per90",  # tacles
            "Tacles réussis /90": "tacklesWon_per90",  # tacles réussis
            "Interceptions /90": "interceptions_per90",  # interceptions
            "Dégagements /90": "clearances_per90",  # dégagements
            "Tirs contrés /90": "blockedShots_per90",  # tirs contrés
            "Duels aériens /90": "aerialDuelsWon_per90",  # duels aériens gagnés
            "Duels sol /90": "groundDuelsWon_per90",  # duels au sol gagnés
            "Passes réussies /90": "accuratePasses_per90",  # passes réussies
            "Dribbles réussis /90": "successfulDribbles_per90",  # dribbles
            "Passes clés /90": "keyPasses_per90",  # passes menant à un tir
            "Touches /90": "touches_per90",  # touches de balle
        },
    },
    "Gardien": {  # gardiens
        "code": "G",  # code du poste chez Sofascore
        "radar": {  # axes du radar
            "Arrêts /90": "saves_per90",  # arrêts
            "Buts encaissés /90": "goalsConceded_per90",  # buts encaissés, une valeur haute est défavorable
            "Penalties arrêtés /90": "penaltySave_per90",  # penalties arrêtés
            "Clean sheets": "cleanSheet",  # matchs sans but encaissé, total de la saison
        },
    },
}

print({poste: len(cfg["radar"]) for poste, cfg in POSTES_RADAR.items()})  # contrôle : on attend 8 axes pour les attaquants, 16 pour les milieux, 15 pour les défenseurs et 4 pour les gardiens

{'Attaquant': 8, 'Milieu': 16, 'Défenseur': 15, 'Gardien': 4}


## 16. Export des données pour le site

Le site n'a besoin ni de Python ni d'une base de données : tout tient dans un fichier de données que la page lit directement. On y met, pour chaque poste :
1. la liste des joueurs triés par nom, avec leur équipe, leur année de naissance, leurs minutes, leurs matchs, leurs titularisations et toutes leurs statistiques ;
2. les valeurs maximales de chaque axe du radar, pour ramener chaque joueur sur 100 ;
3. la matrice complète des pourcentages de ressemblance du poste, ce qui permet d'afficher la ressemblance entre deux joueurs choisis librement, et pas seulement les 10 plus proches.

**Contrôles attendus** : 339 joueurs, des matrices carrées (75, 135, 105 et 24 joueurs par poste), et une ressemblance de 97,7 % entre Lautaro Martínez et Marcus Thuram.

In [22]:
matrices = {  # matrice de pourcentage de chaque poste (calculée à la partie 13)
    "Attaquant": pourcentage_attaque,  # attaquants
    "Milieu": pourcentage_milieu,  # milieux
    "Défenseur": pourcentage_defense,  # défenseurs
    "Gardien": pourcentage_gardien,  # gardiens
}
colonnes_identite_export = ["id", "name", "position", "dateOfBirth", "team", "poste_large"]  # colonnes qui ne sont pas des statistiques
colonnes_stats_export = [c for c in df_filtre.columns if c not in colonnes_identite_export and pd.api.types.is_numeric_dtype(df_filtre[c])]  # toutes les statistiques numériques (totaux et par 90 minutes)

donnees_site = {"saison": "2025/26", "seuil_minutes": SEUIL_MINUTES, "postes": {}}  # contenu du fichier : un bloc par poste

for poste, cfg in POSTES_RADAR.items():  # parcourt les quatre postes
    df_poste = df_filtre[df_filtre["poste_large"] == poste].sort_values("name")  # joueurs du poste, triés par nom
    noms = df_poste["name"].tolist()  # liste des noms dans l'ordre d'affichage
    matrice = matrices[poste].loc[noms, noms]  # matrice de ressemblance remise dans le même ordre que la liste des noms
    colonnes_radar = list(cfg["radar"].values())  # colonnes utilisées par les axes du radar

    joueurs = []  # liste des joueurs de ce poste
    for _, ligne in df_poste.iterrows():  # parcourt les joueurs un par un
        stats = {c: round(float(ligne[c]), 3) for c in colonnes_stats_export if pd.notna(ligne[c])}  # statistiques du joueur, sans les valeurs vides (par exemple le xG des gardiens)
        joueurs.append({  # fiche du joueur
            "nom": ligne["name"],  # nom complet
            "equipe": ligne["team"],  # équipe en 2025/26
            "annee": int(str(ligne["dateOfBirth"])[:4]),  # année de naissance, lue dans la date de naissance
            "minutes": int(ligne["minutesPlayed"]),  # minutes jouées
            "matchs": int(ligne["appearances"]),  # matchs joués
            "titularisations": int(ligne["matchesStarted"]),  # matchs commencés
            "stats": stats,  # toutes les statistiques
        })

    donnees_site["postes"][poste] = {  # bloc du poste
        "code": cfg["code"],  # code Sofascore du poste
        "radar": [[nom, col] for nom, col in cfg["radar"].items()],  # axes du radar : nom affiché et colonne
        "maxima": {c: round(float(pd.to_numeric(df_poste[c]).max()), 3) for c in colonnes_radar},  # valeur maximale de chaque axe dans le poste
        "defaut": int(np.argmax(df_poste["minutesPlayed"].to_numpy())),  # joueur affiché au départ : celui qui a le plus joué
        "joueurs": joueurs,  # fiches des joueurs
        "ressemblance": matrice.round(1).values.tolist(),  # pourcentages de ressemblance entre tous les joueurs du poste
    }

os.makedirs("docs", exist_ok=True)  # crée le dossier du site s'il n'existe pas encore
with open("docs/donnees_joueurs.json", "w", encoding="utf-8") as f:  # fichier de données, gardé à part pour pouvoir le relire
    json.dump(donnees_site, f, ensure_ascii=False, separators=(",", ":"))  # écrit les données en format compact

nb_joueurs = sum(len(p["joueurs"]) for p in donnees_site["postes"].values())  # nombre total de joueurs exportés
print("Joueurs exportés :", nb_joueurs, "| attendu : 339")  # contrôle : on attend 339
print({p: (len(b["joueurs"]), len(b["ressemblance"]), len(b["ressemblance"][0])) for p, b in donnees_site["postes"].items()})  # contrôle : joueurs, lignes et colonnes de la matrice identiques pour chaque poste
att = donnees_site["postes"]["Attaquant"]  # bloc des attaquants
noms_att = [j["nom"] for j in att["joueurs"]]  # noms des attaquants
print("Lautaro Martínez et Marcus Thuram :", att["ressemblance"][noms_att.index("Lautaro Martínez")][noms_att.index("Marcus Thuram")], "| attendu : 97.7")  # contrôle de la ressemblance
print("Taille du fichier :", round(os.path.getsize("docs/donnees_joueurs.json") / 1024), "ko")  # taille du fichier de données

Joueurs exportés : 339 | attendu : 339
{'Attaquant': (75, 75, 75), 'Milieu': (135, 135, 135), 'Défenseur': (105, 105, 105), 'Gardien': (24, 24, 24)}
Lautaro Martínez et Marcus Thuram : 97.7 | attendu : 97.7
Taille du fichier : 648 ko


## 17. Construction du site

Le site est une seule page HTML, sans serveur. On injecte les données dans un modèle de page (`docs/modele.html`), et le fichier produit (`docs/index.html`) s'ouvre dans n'importe quel navigateur. On y choisit un poste et un joueur, on regarde son radar et ses statistiques, on peut le comparer à un deuxième joueur du même poste, et on voit ses 10 profils les plus proches.

Le modèle `docs/modele.html` doit être présent dans le dossier `docs` avant de lancer la cellule.

In [23]:
with open("docs/modele.html", encoding="utf-8") as f:  # ouvre le modèle de page (à placer dans le dossier docs)
    modele = f.read()  # lit tout le modèle
assert "/*DONNEES*/null" in modele, "Le marqueur de données est introuvable dans le modèle"  # arrête avec un message clair si le mauvais fichier est utilisé

json_texte = json.dumps(donnees_site, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")  # données en texte, avec protection de la balise de fin de script
page = modele.replace("/*DONNEES*/null", json_texte)  # insère les données dans la page
with open("docs/index.html", "w", encoding="utf-8") as f:  # écrit la page finale
    f.write(page)  # sauvegarde
print("Page créée : docs/index.html |", round(os.path.getsize("docs/index.html") / 1024), "ko")  # contrôle de la taille du fichier

Page créée : docs/index.html | 667 ko


## Annexe facultative : export vers une base MySQL

Version de départ du projet : les résultats étaient exportés vers une base MySQL, consultée ensuite avec phpMyAdmin, avec des vues SQL par poste (non incluses dans ce dépôt). Cette annexe ne s'exécute pas par défaut, car elle demande une base MySQL installée sur la machine. Pour la lancer, il faut passer `EXPORT_MYSQL` à `True` dans la cellule suivante et adapter l'utilisateur, le mot de passe et le port.

Deux tables sont créées.

1. `joueurs` : la table complète des statistiques.
2. `similarites` : une table longue (joueur, joueur comparé, pourcentage de ressemblance, poste), qui permet de filtrer et trier facilement.

In [24]:
EXPORT_MYSQL = False  # passer à True pour lancer l'annexe, qui demande une base MySQL locale

In [25]:
if EXPORT_MYSQL:  # annexe facultative
    from sqlalchemy import create_engine  # gère la connexion à la base MySQL

    utilisateur = "root"  # nom d'utilisateur MySQL
    mot_de_passe = ""  # mot de passe MySQL
    hote = "localhost"  # adresse du serveur MySQL
    port = 3307  # port MySQL (à adapter selon l'installation, vérifiable dans le panneau de contrôle XAMPP/WAMP/MAMP)
    nom_base = "recruefoot"  # nom de la base créée au préalable dans phpMyAdmin

    url_connexion = f"mysql+pymysql://{utilisateur}:{mot_de_passe}@{hote}:{port}/{nom_base}"  # URL de connexion complète

    engine = create_engine(url_connexion)  # objet de connexion, réutilisable pour les exports suivants

    with engine.connect() as connexion:  # test de connexion
        print("Connexion réussie !")  # confirme que la connexion fonctionne

In [26]:
if EXPORT_MYSQL:  # annexe facultative
    def matrice_vers_table_longue(matrice_pourcentage, poste):  # petite recette : transforme une matrice carrée en table longue
        # matrice_pourcentage : DataFrame carré (joueur x joueur) avec les scores de ressemblance
        # poste : nom du poste concerné, ajouté comme colonne pour garder la trace du groupe d'origine
        matrice_renommee = matrice_pourcentage.rename_axis(index="joueur", columns="joueur_compare")  # noms différents pour l'index et les colonnes, pour éviter un conflit lors du reset_index()
        table_longue = matrice_renommee.stack().reset_index()  # transforme la matrice carrée en 3 colonnes : joueur, joueur_compare, valeur
        table_longue.columns = ["joueur", "joueur_compare", "pourcentage_ressemblance"]  # renomme la 3e colonne générée par stack()
        table_longue = table_longue[table_longue["joueur"] != table_longue["joueur_compare"]]  # retire les lignes où un joueur est comparé à lui-même
        table_longue["poste"] = poste  # ajoute le poste
        return table_longue  # renvoie la table longue de ce poste

    table_similarites = pd.concat([  # empile les 4 tables longues (une par poste) en une seule table finale
        matrice_vers_table_longue(pourcentage_attaque, "Attaquant"),  # table longue des attaquants
        matrice_vers_table_longue(pourcentage_milieu, "Milieu"),  # table longue des milieux
        matrice_vers_table_longue(pourcentage_defense, "Défenseur"),  # table longue des défenseurs
        matrice_vers_table_longue(pourcentage_gardien, "Gardien"),  # table longue des gardiens
    ], ignore_index=True)  # index propre plutôt que des index dupliqués entre les 4 morceaux

    print(table_similarites.shape)  # contrôle : on attend 35 112 lignes et 4 colonnes

In [27]:
if EXPORT_MYSQL:  # annexe facultative
    df_filtre.to_sql("joueurs", con=engine, if_exists="replace", index=False)  # exporte la table complète des stats ; if_exists="replace" écrase la table si elle existe déjà (utile pour relancer après une mise à jour des données)

    table_similarites.to_sql("similarites", con=engine, if_exists="replace", index=False)  # exporte la table longue de similarité

    print("Export terminé")  # confirme la fin de l'export

## Limites et précautions de lecture

1. **La ressemblance compare des profils statistiques, pas la qualité.** Deux joueurs à 95 % peuvent avoir des niveaux très différents : le score dit « ils jouent de la même façon », pas « ils se valent ».
2. **Les postes sont larges.** Un ailier et un avant centre sont tous deux des attaquants. Un utilisateur doit regarder la fiche avant de conclure.
3. **Le contexte d'équipe pèse sur les chiffres.** Un milieu d'une équipe qui domine fait plus de passes qu'un milieu d'une équipe qui défend. Le système ne corrige pas cet effet.
4. **Seuil de 900 minutes.** Les joueurs qui ont peu joué (jeunes, remplaçants) sont absents. La saison étudiée est 2025/26, en Serie A uniquement.
5. **Une seule source.** Toutes les statistiques viennent de Sofascore, y compris xG et xA, qui sont les modèles de Sofascore.
6. **Gardiens.** Seuls 24 gardiens sont comparés. Pour les buts encaissés, une valeur haute est défavorable. Les clean sheets sont un total de saison, qui dépend du nombre de matchs joués.
7. **Le radar est relatif.** Chaque axe est ramené sur 100 par rapport au meilleur joueur du poste. Un joueur à 100 sur un axe est le meilleur de son poste, pas un joueur parfait.